In [ ]:
!pip install pymupdf nltk

import pymupdf
import nltk
import re

!pip install xlsxwriter
import xlsxwriter

!pip install openai
from openai import OpenAI

import os
from pydantic import BaseModel

import pandas as pd
os.environ["OPENAI_API_KEY"] = #INSERT OpenAI API KEY HERE"

#https://www.nltk.org/api/nltk.tokenize.html
nltk.download('punkt_tab')

#splits on on whitespace and punctuation, e.g.: AI-driven = "AI" "-" "driven"
from nltk.tokenize import wordpunct_tokenize
from nltk.util import bigrams

from pathlib import Path

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/sebastianronan/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
keywords = {"ai", "genai", "ml", "agentic", "ki", "agentisk", "maskinlæring"}
#English: "ai", "genai", "ml", "agentic"
#Danish: "ki", "genai", "ml", "maskinlæring", "agentisk"

bigram_keywords = {("gen", "ai"), ("artificial", "intelligence"), ("machine", "learning"),("generative", "ai"),
 ("generativ", "ai"), ("generativ", "ki"), ("kunstig", "intelligens"), ("maskin", "læring")}
#English: ("gen", "ai"), ("artificial", "intelligence"), ("machine", "learning"), ("generative", "ai")
#Danish: ("generativ", "ai"), ("generativ", "ki"), ("kunstig", "intelligens"), ("maskin", "læring"),

ai_keyword = {"ai", "ki"}
ai_bigram = {("artificial", "intelligence"), ("kunstig", "intelligens")}

##setting up prompt
prompt = ("""
Extract the following information from the annual report:
- The company's legal name
- The reporting/financial year of the annual report
""")

##setting up LLM
client = OpenAI()

#Structure for its output
class TextAnalysis(BaseModel):
  company_name: str
  reporting_year: int


#https://docs.python.org/3/library/pathlib.html
companies = list(Path("Missing Reports").rglob("*.pdf"))


#rows int the panda
rows = []

#start looping the companies
for file in companies:

#https://pymupdf.readthedocs.io/en/latest/tutorial.html
#https://pymupdf.readthedocs.io/en/latest/app1.html
  #extract text as string
    doc = pymupdf.open(file)

    str_text = ""
    for page in doc:
      str_text += page.get_text("text")

    #https://www.nltk.org/api/nltk.tokenize.html
    #https://www.geeksforgeeks.org/python/python-string-lower/
    tokens_str = [t.lower() for t in wordpunct_tokenize(str_text)]
    tokens_bi = list(bigrams(tokens_str))

    #count share of ai
    count_1 = sum(tokens_str.count(a) for a in ai_keyword)
    count_2 = sum(tokens_bi.count(b) for b in ai_bigram)
    count = count_1 + count_2
    ai_share = (count / len(tokens_str))*100 if tokens_str else 0 # To avoid crashes


    #create shorter string for the LLM to extract name and year
    begin_end_text = str_text[:1000] + str_text[-1000:]

    #https://developers.openai.com/api/docs/guides/structured-outputs?example=structured-data
    #run LLM
    response = client.responses.parse(
        model="gpt-4o-mini",
        temperature=0,
        input=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": begin_end_text},
        ],
        text_format=TextAnalysis,
    )
    print(f"{file.name} complete")  # prints after each report is processed to ensure that we know what's going on

    #LLM result saved back to the list
    parsed = response.output_parsed


    #save text as paragraphs for dict
    paragraphs = []

    for page in doc:
        for b in page.get_text("blocks"):
            text = b[4]
            if text:
                paragraphs.append(text)


    #list to save paragraphs with a keyword match
    relevant_paragraphs = []

    #tokenize the paragraphs into single tokens and bigrams
    for paragraph in paragraphs:
        tokens = [t.lower() for t in wordpunct_tokenize(paragraph)]
        token_bigrams = set(bigrams(tokens))

        #https://www.geeksforgeeks.org/python/intersection-function-python/

        #save paragraph to list if it includes the keywords and tokens or bigrams are the same
        if keywords.intersection(tokens) or bigram_keywords.intersection(token_bigrams):
            relevant_paragraphs.append(paragraph)


   #add all the saved lists to the dataframe
    rows.append(
    {"File": file,
     "Company Name": parsed.company_name,
     "Year": parsed.reporting_year,
     "AI share": ai_share,
     "AI Paragraph": relevant_paragraphs,
     }
    )

#https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.to_csv.html

df = pd.DataFrame(rows)
df.to_csv("full_100_2.csv", sep=';', index = True, header = True)

print("Done")  # prints once all reports are finished

FirstFarms_Aarsrapport_2021_PDF_version.pdf complete
FirstFarms_Aarsrapport_2024.pdf complete
FirstFarms_Aarsrapport_2025.pdf complete
FirstFarms_AArsrapport_2022.pdf complete
FirstFarms_Aarsrapport_2023.pdf complete
AArsrapport_2020.pdf complete
Strategic-Partners-AS-2024-AR.pdf complete
SP-2025-AnnualReport_final-2.pdf complete
Annual-Report-2020.pdf complete
Annual-Report-2022.pdf complete
Orphazyme-2023-arsrapport.pdf complete
SP-2025-AnnualReport_final.pdf complete
Årsrapport 2024-2025.pdf complete
Aarsrapport-2020-21-Brd.-Klee.pdf complete
Årsrapport 2022-23 Klee og koncern.pdf complete
Årsrapport 2023-24.pdf complete
Aarsrapport-2021-22-1.pdf complete
AR2020.pdf complete
AR2021.pdf complete
AR2023.pdf complete
AR2022.pdf complete
AR2025.pdf complete
AR2024.pdf complete
prime-office-as-arsregnskab-2022.pdf complete
prime-office-aarsregnskab-2020.pdf complete
Aarsregnskab-2025.pdf complete
prime-office-2021_ny1.pdf complete
prime-office-arsregnskab-2023.pdf complete
prime-offic

Annual Report 2022.pdf complete
Annual Report 2023.pdf complete
Annual Report 2021.pdf complete
annual-report-2020 (2).pdf complete
Annual Report 2024 (1).pdf complete
columbus-annual-report-2025.pdf complete
APMM-AR-2022.pdf complete
APMM Annual Report 2023.pdf complete
apmm-ar-2025-70-final-no-tags.pdf complete
Maersk Annual Report 2024.pdf complete
Maersk Annual Report 2020.pdf complete
APMM AR 2021_20220209 no iXBRL.pdf complete
eac-invest-arsrapport-2023.pdf complete
eac-invest-arsrapport-2022.pdf complete
eac-invest-arsrapport-2020.pdf complete
eac-invest-arsrapport-2021.pdf complete
ok-arsrapport-2024.pdf complete
aarsrapport-2023-til-web.pdf complete
djurslands-bank-rsrapport-2022-web.pdf complete
djurslands-bank-aarsrapport-2025.pdf complete
aarsrapport-2020.pdf complete
aarsrapport-2021-online.pdf complete
aarsrapport_2024 (1).pdf complete
Annual report 2023-3.pdf complete
Annual report 2021.pdf complete
Annual report 2024-2.pdf complete
Annual report 2020-3.pdf complete
TCM 

Asetek Annual Report 2021.pdf complete
Asetek Annual Report 2020.pdf complete
Asetek_annual_report_2023.pdf complete
Asetek-Annual-Report-2022wu3.pdf complete
B&O_Annual report 2022-23.pdf complete
B&O_Annual Report 2020-21.pdf complete
Annual Report 2021_22n.pdf complete
B&O_Annual report 2024-25.pdf complete
B&O_Annual report 2023-24.pdf complete
Annual-report_24-25.pdf complete
Annual-report-2023-2024.pdf complete
Annual-report-2022-2023.pdf complete
Aarsrapport-2020-2021.pdf complete
Aarsrapport-2021-2022.pdf complete
virogates-annual-report-2021.pdf complete
virogates-annual-report-2020.pdf complete
virogates-annual-report-2022.pdf complete
virogates-annual-report-2023.pdf complete
20250320-annual-report-2024.pdf complete
LSB_aarsrapport_2021.pdf complete
LSB_aarsrapport_2025.pdf complete
regnskab2020.pdf complete
aarsrapport_2024.pdf complete
aarsrapport_2022.pdf complete
aarsrapport_2023.pdf complete
NORDEN_AnnualReport_2023.pdf complete
NORDEN Annual Report 2025.pdf complete
NO

Annual Report 2025.pdf complete
Annual Report 2020 (1).pdf complete
NTG_Annual_Report_2021_WEB.pdf complete
NTG Annual Report 2022.pdf complete
NTG Annual Report 2023.pdf complete
2025-staticcontent.pdf complete
2024-staticcontent-3.pdf complete
2022-staticcontent-5.pdf complete
2023-staticcontent-4.pdf complete
2020-staticcontent-7.pdf complete
2021-staticcontent-6.pdf complete
Aarsleff_AR-2021-22_UK.pdf complete
Aarsleff_Annual_Report_2020-21_UK.pdf complete
Aarsleff_annual report-2024-25_UK.pdf complete
Aarsleff_AR-2022-23_UK_FINAL.pdf complete
Aarsleff_AR-2023-24_UK_final_web.pdf complete
amNsb3VkczovLzAzLzhlLzM3LzNhLzQ3L2IzZGQtNGQ0NS1hODk5LTgwOTM5MDkwNjFjMw.pdf complete
bif-2025-06-30-da.pdf complete
fbm-nr-02-2021-brondby-if-aarsrapport-2020.pdf complete
aarsrapport-2021-2.pdf complete
a-rsrapport-2022.pdf complete
2022-Danske Bank Annual report 2022.pdf complete
2025-Danske Bank - Annual Report 2025.pdf complete
2023-Danske Bank - Annual Report 2023.pdf complete
2020-Annual Repo

Copenhagen-Capital-Aarsrapport-2025.pdf complete
Selskabsmeddelelse-Arsrapport-2020.pdf complete
Arsrapport_2022_final.pdf complete
2-meddelelse-2025.pdf complete
db1334a8-ec0e-4802-b15b-383f0db5f9b9.pdf complete
3_ýrsrapport-2023.pdf complete
590b2679-5a08-407b-9f06-3db4314d9adc.pdf complete
21_aarsrapport-2026.pdf complete
2-aarsrapport-2023.pdf complete
2025-Annual-Report.pdf complete
2021-Monsenso-Annual-report.pdf complete
2023-Annual-Report.v3.pdf complete
230308.Monsenso.2022.Annual.Report.v1.pdf complete
250307.Annual.Report.2024.v4.pdf complete
Annual-report-2020.Monsenso.pdf complete
hh-international-as2021-annual-report.pdf complete
hh_ar-2025.pdf complete
annual-report-2023 (5).pdf complete
annual_report_2022.pdf complete
annual-report-2024 (3).pdf complete
hh-international-asannual-report-2020.pdf complete
2022-novo-nordisk-annual-report-2022.pdf complete
2021-novo-nordisk-annual-report-2021.pdf complete
2025-novo-nordisk-annual-report-2025.pdf complete
2024-novo-nordisk-a

In [ ]:
df = pd.read_csv('full_100_2.csv', sep=';')

prompt = ("""
You are assisting in evaluating firms' dynamic capabilities (sensing, seizing, transforming)
in their adoption of Artificial Intelligence (AI).

Dynamic Capabilities describe how organisations sense, act on, and adapt to rapidly changing business environments by continuously developing and reconfiguring their internal and external resources and competences.They are uniquely developed within each firm through its resources, learning, and history, and cannot easily be bought or imitated.

Sensing: The capability to identify and assess opportunities and threats in the external environment through market scanning, customer feedback, technology scouting, and broader ecosystem monitoring, complemented by internal R&D efforts.
Seizing: The capability to mobilise resources and investments to capture sensed opportunities by strategically prioritising competing investment paths and aligning the chosen direction with overall business objectives.
Transforming: The capability to restructure and realign organisational processes, assets, and business models in response to change, including renewed ecosystem relationships, resource reallocation, and staff upskilling.

AI Dynamic Capabilities refer to an organisation's ability to integrate, build, and reconfigure internal and external resources and competences to adopt and leverage artificial intelligence in rapidly changing business environments.

AI Sensing: The capability to detect and interpret AI advancements in the external environment through market scanning, technology scouting, and information collection, complemented by internal AI R&D efforts to identify shifting customer preferences and inform strategic adjustments.
AI Seizing: The capability to capture value from sensed AI opportunities by prioritising the piloting and development of AI-driven products and innovation, mobilising resources accordingly, and promoting an organisational culture that embraces AI integration aligned with overall business objectives.
AI Transforming: The capability to restructure the organisation to exploit AI opportunities, spanning internal transformation, such as staff upskilling, shifting toward AI-driven business models, and patenting AI-enabled technology, and external transformation through renewed partnerships with universities and industry partners.

SCORING PROCEDURE

1. Identify explicit textual evidence for sensing, seizing, and transforming.
2. Determine which rubric level the evidence satisfies.
3. If evidence is ambiguous between two adjacent scores, assign the LOWER score.
4. If explicit evidence is absent, assign the lowest applicable score.

Do not infer capabilities beyond what is explicitly stated in the text.

DATA NOTE:

The provided text contains ONLY paragraphs mentioning AI from annual reports.
Therefore the density of AI discussion is artificially high.

The mere presence of AI language should NOT increase scores.
Only structural evidence of capabilities should affect scoring.


SCORING RUBRIC:

Sensing:
0: AI is not mentioned at all in a sensing context.
1: AI is mentioned without evidence of monitoring or analysis.
2: AI is acknowledged as a general trend but without firm-specific monitoring.
3: AI trends are discussed with data or references, suggesting external monitoring,
but no clear strategic interpretation for the firm.
4: Structured and recurring monitoring of AI developments is demonstrated through
multiple indicators (data, partnerships, research references, technology scouting),
with partial discussion of firm-level implications.
5: Systematic, multi-source environmental scanning with dedicated intelligence
mechanisms such as internal research units or structured technology scouting.

Seizing:
0: AI is not mentioned at all in a seizing context.
1: AI mentioned but too vague to identify concrete capability or investment.
2: A specific AI initiative exists but is minor, isolated, or experimental.
3: A named AI system, platform, or tool exists but scope and strategic integration
are unclear.
4: AI capabilities are embedded in clearly described products or operational systems
with identifiable business objectives and multiple coordinated initiatives
implemented at meaningful organisational scale.
5: AI capabilities form a central pillar of the firm's delivery model and value
proposition with extensive investment across products, operations, workforce,
and strategic planning.

Transforming:
0: AI is not mentioned at all in a transforming context.
1: Organizational change mentioned but AI plays no meaningful role.
2: AI-related organisational change referenced but vague.
3: Specific AI-related organisational adjustment described but limited in scope.
4: Substantial organisational adaptation including workforce upskilling,
process redesign, or governance mechanisms at significant scale.
5: Comprehensive organisational transformation including structural redesign,
organisation-wide capability building, governance frameworks, and ecosystem alignment.

""")

client = OpenAI()


class TextAnalysis(BaseModel):
    AI_sensing_score: int
    AI_seizing_score: int
    AI_transforming_score: int


AI_sensing_score = []
AI_seizing_score = []
AI_transforming_score = []


for i, t in enumerate(df['AI Paragraph'].fillna("")):  # added enumerate to track row number
    response = client.responses.parse(
        model="gpt-4.1-mini",
        temperature=0,
        input=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": t},
        ],
        text_format=TextAnalysis,
    )

    parsed = response.output_parsed

    AI_sensing_score.append(parsed.AI_sensing_score)
    AI_seizing_score.append(parsed.AI_seizing_score)
    AI_transforming_score.append(parsed.AI_transforming_score)

    print(f"{df['Company Name'].iloc[i]} ({df['Year'].iloc[i]}) complete")  # prints after each company is scored


df['Sensing'] = AI_sensing_score
df['Seizing'] = AI_seizing_score
df['Transforming'] = AI_transforming_score

df.to_csv('Missing reports.csv', index=False)

print("Done")  #prints once all companies are scored

FirstFarms A/S (2021) complete
FirstFarms A/S (2024) complete
FirstFarms A/S (2025) complete
FirstFarms A/S (2022) complete
FirstFarms A/S (2023) complete
FirstFarms A/S (2020) complete
[Extracted Company Name] (2023) complete
Strategic Partners A/S (2025) complete
Orphazyme A/S (2020) complete
Orphazyme A/S (2022) complete
Orphazyme A/S (2023) complete
Strategic Partners A/S (2025) complete
Brd. Klee A/S (2025) complete
BRD. KLEE A/S (2021) complete
Brd. Klee A/S (2023) complete
Brd. Klee A/S (2024) complete
BRD. KLEE A/S (2022) complete
Agillic (2020) complete
Agillic A/S (2021) complete
Agillic A/S (2023) complete
Agillic A/S (2022) complete
Agillic A/S (2025) complete
Agillic A/S (2024) complete
Prime Office A/S (2022) complete
Prime Office A/S (2020) complete
PRIME OFFICE A/S (2025) complete
PRIME OFFICE A/S (2021) complete
Prime Office A/S (2023) complete
PRIME OFFICE A/S (2024) complete
Done
